In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/College/FLEX-Med/flex-med

In [ ]:
!pip install nest-asyncio uvicorn pyngrok fastapi torch torchvision pytorch-grad-cam lime scikit-image

# **⚠️ IMPORTANT: Cleanup Existing ngrok Sessions**

**Problem**: Free ngrok accounts are limited to 1 simultaneous tunnel.

**Solution**: Run the cell below to kill ALL existing ngrok sessions before starting this server.

**Note**: If you have the FL orchestrator running (`start_fl.ipynb`), you'll need to stop it first or upgrade your ngrok plan.

In [ ]:
import subprocess
import time
from pyngrok import ngrok
from pyngrok.process import kill_process

print("🧹 Cleaning up existing ngrok sessions...")

try:
    # Method 1: Disconnect all ngrok tunnels via pyngrok
    print("  → Disconnecting all tunnels via pyngrok...")
    ngrok.kill()
    time.sleep(1)
except Exception as e:
    print(f"  ⚠️ pyngrok cleanup: {e}")

try:
    # Method 2: Kill ngrok process
    print("  → Killing ngrok process...")
    subprocess.run(["pkill", "-9", "ngrok"], check=False)
    time.sleep(1)
except Exception as e:
    print(f"  ⚠️ pkill cleanup: {e}")

try:
    # Method 3: Kill port 8000
    print("  → Killing processes on port 8000...")
    subprocess.run(["fuser", "-k", "8000/tcp"], check=False)
    time.sleep(1)
except Exception as e:
    print(f"  ⚠️ Port cleanup: {e}")

print("✅ Cleanup complete! Ready to start server.\n")
print("⚠️  If you still get ERR_NGROK_108, you must:")
print("    1. Stop the FL orchestrator notebook (start_fl.ipynb)")
print("    2. OR upgrade to ngrok paid plan for multiple tunnels")

In [ ]:
import nest_asyncio
import uvicorn
import threading
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import get_model, list_models
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import os
import io
import base64
import datetime
import traceback
import numpy as np
from PIL import Image
from pyngrok import ngrok
from pyngrok.process import kill_process
from pyngrok.exception import PyngrokNgrokError
from fastapi import FastAPI, HTTPException, BackgroundTasks, File, UploadFile
from fastapi.responses import FileResponse, PlainTextResponse, JSONResponse
from pydantic import BaseModel
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from lime import lime_image
from skimage.segmentation import mark_boundaries

# ---------------------------------------------------------
# 0. Setup & Cleanup
# ---------------------------------------------------------
try:
    ngrok.disconnect()
    kill_process()
except:
    pass

app = FastAPI()
nest_asyncio.apply()

# ---------------------------------------------------------
# 1. UNIFIED LOGGING SYSTEM
# ---------------------------------------------------------
CENTRAL_LOG_FILE = "local_training.log"
training_logs = {}

def write_to_central_log(message, level="INFO"):
    """
    Writes to the central local_training.log file.
    Similar to simulation_run.log for federated learning.
    """
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    formatted_msg = f"[{timestamp}] [{level}] {message}"
    
    # Write to central log file
    with open(CENTRAL_LOG_FILE, "a") as f:
        f.write(formatted_msg + "\n")
        f.flush()  # Force write immediately
    
    # Also print to console
    print(formatted_msg)

def log_message(client_name, message, level="INFO"):
    """
    Enhanced logging that writes to:
    1. Central log file (local_training.log)
    2. Per-client log file ({client_name}.log)
    3. In-memory storage for API queries
    4. Console output
    """
    timestamp = datetime.datetime.now().strftime("%H:%M:%S")
    short_msg = f"[{timestamp}] {message}"
    full_msg = f"[{client_name}] {short_msg}"

    # 1. Write to CENTRAL log (like simulation_run.log)
    write_to_central_log(f"[{client_name}] {message}", level)

    # 2. Write to per-client log
    client_log_file = f"{client_name}.log"
    with open(client_log_file, "a") as f:
        f.write(short_msg + "\n")
        f.flush()  # Force write immediately

    # 3. Update in-memory logs
    if client_name not in training_logs:
        training_logs[client_name] = {"status": "starting", "logs": []}
    training_logs[client_name]["logs"].append(short_msg)

# Initialize central log file
with open(CENTRAL_LOG_FILE, "w") as f:
    f.write(f"===== LOCAL TRAINING ORCHESTRATOR LOG =====\n")
    f.write(f"Started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"==========================================\n\n")

write_to_central_log("Local Training Orchestrator initialized")

# ---------------------------------------------------------
# 2. Data Models
# ---------------------------------------------------------

class ClientEnrollment(BaseModel):
    client_name: str
    model_type: str
    num_classes: int = 2

class TrainRequest(BaseModel):
    client_name: str
    dataset_path: str
    model_path: str
    epochs: int = 5
    batch_size: int = 16
    learning_rate: float = 0.001

# ---------------------------------------------------------
# 3. Model Architecture Helper
# ---------------------------------------------------------

def modify_last_layer(model, num_classes):
    if num_classes == 1000: return model

    if hasattr(model, 'fc') and isinstance(model.fc, nn.Linear):
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Linear):
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Sequential):
        updated = False
        for i in range(len(model.classifier) - 1, -1, -1):
            if isinstance(model.classifier[i], nn.Linear):
                model.classifier[i] = nn.Linear(model.classifier[i].in_features, num_classes)
                updated = True
                break
        if not updated:
             model.classifier.add_module("final_fc", nn.Linear(1280, num_classes))
    return model

# ---------------------------------------------------------
# 4. Common Transform (ImageNet Normalization)
# ---------------------------------------------------------

COMMON_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ---------------------------------------------------------
# 5. XAI Helper Functions
# ---------------------------------------------------------

def unnormalize_tensor(tensor: torch.Tensor) -> np.ndarray:
    """
    Input: (3, H, W) normalized tensor
    Output: (H, W, 3) numpy image in [0, 1]
    """
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    img = tensor.detach().cpu() * std + mean
    img = img.permute(1, 2, 0).numpy()
    return np.clip(img, 0, 1)

def generate_gradcam_base64(
    model: torch.nn.Module,
    input_tensor: torch.Tensor,
    target_layers: list,
    class_idx: int,
) -> str:
    """
    Generate Grad-CAM and return base64 PNG.
    input_tensor: (1, 3, H, W)
    """
    cam = GradCAM(model=model, target_layers=target_layers)

    targets = [ClassifierOutputTarget(class_idx)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    rgb_img = unnormalize_tensor(input_tensor[0])
    cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    pil_img = Image.fromarray(cam_image)
    buffer = io.BytesIO()
    pil_img.save(buffer, format="PNG")

    return base64.b64encode(buffer.getvalue()).decode("utf-8")

def generate_lime_base64(
    model: torch.nn.Module,
    input_tensor: torch.Tensor,
    class_idx: int,
    device: torch.device,
    num_samples: int = 1000,
) -> str:
    """
    Generate LIME explanation and return base64 PNG.
    """

    # LIME expects numpy image in [0,255]
    img = unnormalize_tensor(input_tensor[0])
    img_uint8 = (img * 255).astype(np.uint8)

    def predict_fn(images):
        """
        images: List[np.ndarray] of shape (H, W, 3)
        returns: probabilities
        """
        tensors = torch.stack(
            [
                COMMON_TRANSFORM(Image.fromarray(im)).to(device)
                for im in images
            ]
        )

        with torch.no_grad():
            outputs = model(tensors)
            probs = torch.softmax(outputs, dim=1)

        return probs.cpu().numpy()

    explainer = lime_image.LimeImageExplainer()

    explanation = explainer.explain_instance(
        img_uint8,
        classifier_fn=predict_fn,
        labels=(class_idx,),
        hide_color=0,
        num_samples=num_samples,
    )

    lime_img, mask = explanation.get_image_and_mask(
        label=class_idx,
        positive_only=True,
        num_features=5,
        hide_rest=False,
    )

    lime_vis = mark_boundaries(lime_img / 255.0, mask)

    pil_img = Image.fromarray((lime_vis * 255).astype(np.uint8))
    buffer = io.BytesIO()
    pil_img.save(buffer, format="PNG")

    return base64.b64encode(buffer.getvalue()).decode("utf-8")

def get_gradcam_target_layers(model: torch.nn.Module, model_name: str):
    """
    Resolve target layers for Grad-CAM based on architecture.
    """
    model_name_lower = model_name.lower()
    
    if "resnet" in model_name_lower:
        return [model.layer4[-1]]
    elif "mobilenet" in model_name_lower:
        return [model.features[-1]]
    elif "efficientnet" in model_name_lower:
        return [model.features[-1]]
    elif "densenet" in model_name_lower:
        return [model.features[-1]]
    elif "vgg" in model_name_lower:
        return [model.features[-1]]
    else:
        # Default fallback - try to find the last convolutional layer
        try:
            if hasattr(model, 'features'):
                return [model.features[-1]]
            elif hasattr(model, 'layer4'):
                return [model.layer4[-1]]
        except:
            pass
        raise ValueError(f"Unsupported model for Grad-CAM: {model_name}")

# ---------------------------------------------------------
# 6. Training Logic with VERBOSE Logging
# ---------------------------------------------------------

def run_training_background(client_name: str, dataset_path: str, model_path: str, epochs: int, batch_size: int, lr: float):
    """
    Background Task: Trains the model with detailed batch-level logging.
    """
    write_to_central_log(f"=== Training Started: {client_name} ===")
    log_message(client_name, f"🔄 Starting training for {model_path}...")
    log_message(client_name, f"Config: epochs={epochs}, batch_size={batch_size}, lr={lr}")
    training_logs[client_name]["status"] = "running"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log_message(client_name, f"⚙️ Device: {device}")

    try:
        # 1. Load Checkpoint
        log_message(client_name, "📂 Loading model checkpoint...")
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model path {model_path} does not exist.")

        checkpoint = torch.load(model_path, map_location=device)
        model_name = checkpoint.get('model_type', 'resnet18')
        num_classes = checkpoint.get('num_classes', 2)
        log_message(client_name, f"✓ Checkpoint loaded: {model_name}")

        # 2. Rebuild Model
        log_message(client_name, f"🏗️ Building {model_name} architecture...")
        model = get_model(model_name, weights=None)
        model = modify_last_layer(model, num_classes)
        log_message(client_name, "✓ Architecture built, loading weights...")
        model.load_state_dict(checkpoint['state_dict'])
        model = model.to(device)
        log_message(client_name, f"✓ Model loaded to {device}")

        # 3. Prepare Data
        log_message(client_name, "📦 Preparing dataset transforms...")
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        log_message(client_name, f"📂 Loading images from {dataset_path}...")
        dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)
        log_message(client_name, f"✓ Dataset: {len(dataset)} images, {len(dataloader)} batches")

        # 4. Training Setup
        log_message(client_name, "⚙️ Setting up optimizer and loss function...")
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        log_message(client_name, "✓ Training setup complete, starting training loop...")

        # 5. Training Loop with BATCH-LEVEL logging
        model.train()
        for epoch in range(epochs):
            log_message(client_name, f"{'='*50}")
            log_message(client_name, f"📊 EPOCH {epoch+1}/{epochs} STARTING")
            log_message(client_name, f"{'='*50}")
            
            running_loss = 0.0
            epoch_start = datetime.datetime.now()
            
            for i, (images, labels) in enumerate(dataloader):
                batch_start = datetime.datetime.now()
                
                # Log every batch for debugging
                if i == 0:
                    log_message(client_name, f"   Batch {i+1}/{len(dataloader)}: Moving data to {device}...")
                
                images, labels = images.to(device), labels.to(device)
                
                if i == 0:
                    log_message(client_name, f"   Batch {i+1}: Running forward pass...")
                
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                if i == 0:
                    log_message(client_name, f"   Batch {i+1}: Running backward pass...")
                
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                
                batch_time = (datetime.datetime.now() - batch_start).total_seconds()
                
                # Log first 3 batches in detail, then every 5 batches
                if i < 3 or (i + 1) % 5 == 0:
                    log_message(client_name, f"   ✓ Batch {i+1}/{len(dataloader)}: loss={loss.item():.4f}, time={batch_time:.2f}s")

            # Epoch Stats
            epoch_time = (datetime.datetime.now() - epoch_start).total_seconds()
            avg_loss = running_loss / len(dataloader)
            log_message(client_name, f"📈 Epoch {epoch+1}/{epochs} COMPLETE - Avg Loss: {avg_loss:.4f}, Time: {epoch_time:.1f}s")

        # 6. Save Updated Model
        log_message(client_name, "💾 Saving updated model...")
        checkpoint['state_dict'] = model.state_dict()
        torch.save(checkpoint, model_path)

        training_logs[client_name]["status"] = "completed"
        log_message(client_name, f"✅ Training Complete. Model saved to {model_path}")
        write_to_central_log(f"=== Training Completed: {client_name} ===")

    except Exception as e:
        training_logs[client_name]["status"] = "failed"
        error_msg = f"❌ Training Failed: {str(e)}"
        stack_trace = traceback.format_exc()
        log_message(client_name, error_msg, level="ERROR")
        log_message(client_name, f"Stack trace:\n{stack_trace}", level="ERROR")
        write_to_central_log(f"=== Training Failed: {client_name} - {str(e)} ===", level="ERROR")

# ---------------------------------------------------------
# 7. API Endpoints
# ---------------------------------------------------------

@app.get("/")
def read_root():
    return {
        "status": "Server running",
        "service": "Local Training Orchestrator with xAI",
        "log_file": CENTRAL_LOG_FILE
    }

@app.post("/enroll_client")
async def enroll_client(client: ClientEnrollment):
    write_to_central_log(f"Client enrollment requested: {client.client_name} ({client.model_type})")
    try:
        model_name = client.model_type.lower().strip()
        available_models = list_models()
        if model_name not in available_models:
            raise HTTPException(status_code=400, detail="Model not found.")

        model = get_model(model_name, weights='DEFAULT')
        model = modify_last_layer(model, client.num_classes)

        base_path = "/content/drive/MyDrive/College/models"
        if not os.path.exists(base_path): 
            os.makedirs(base_path, exist_ok=True)
        full_path = f"{base_path}/{client.client_name}.pt"

        checkpoint = {
            'model_type': model_name,
            'num_classes': client.num_classes,
            'state_dict': model.state_dict()
        }
        torch.save(checkpoint, full_path)
        
        write_to_central_log(f"Client enrolled successfully: {client.client_name} -> {full_path}")
        return {"status": "success", "file_path": full_path}
    except Exception as e:
        write_to_central_log(f"Enrollment failed for {client.client_name}: {str(e)}", level="ERROR")
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/start_local_train")
async def start_local_train(request: TrainRequest, background_tasks: BackgroundTasks):
    """
    Starts training and initializes logging for the client.
    """
    write_to_central_log(f"Training request received for: {request.client_name}")
    write_to_central_log(f"  Dataset: {request.dataset_path}")
    write_to_central_log(f"  Model: {request.model_path}")
    write_to_central_log(f"  Config: epochs={request.epochs}, batch_size={request.batch_size}, lr={request.learning_rate}")

    if not os.path.exists(request.dataset_path):
        write_to_central_log(f"Dataset path not found: {request.dataset_path}", level="ERROR")
        raise HTTPException(status_code=404, detail="Dataset path not found.")

    # Initialize per-client log file
    log_file = f"{request.client_name}.log"
    with open(log_file, "w") as f:
        f.write(f"--- Training Log for {request.client_name} ---\n")
        f.write(f"Started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

    training_logs[request.client_name] = {"status": "pending", "logs": ["Request received."]}

    # Add to background
    background_tasks.add_task(
        run_training_background,
        request.client_name,
        request.dataset_path,
        request.model_path,
        request.epochs,
        request.batch_size,
        request.learning_rate
    )

    return {
        "status": "started",
        "message": f"Training started for {request.client_name}",
        "log_files": {
            "central": CENTRAL_LOG_FILE,
            "client": f"{request.client_name}.log"
        }
    }

@app.get("/train_status/{client_name}")
async def get_train_status(client_name: str):
    """
    Returns the logs and status for a specific client.
    """
    if client_name not in training_logs:
        raise HTTPException(status_code=404, detail="No training logs found for this client.")

    return training_logs[client_name]

@app.get("/logs")
async def get_central_logs(lines: int = 100):
    """
    Fetch recent lines from the central local_training.log file.
    Similar to tailing simulation_run.log.
    
    Args:
        lines: Number of recent lines to return (default: 100)
    """
    if not os.path.exists(CENTRAL_LOG_FILE):
        return {"logs": [], "message": "No logs available yet"}
    
    try:
        with open(CENTRAL_LOG_FILE, "r") as f:
            all_lines = f.readlines()
            recent_lines = all_lines[-lines:] if len(all_lines) > lines else all_lines
        
        return {
            "logs": [line.strip() for line in recent_lines],
            "total_lines": len(all_lines),
            "returned_lines": len(recent_lines)
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to read logs: {str(e)}")

@app.get("/logs/download")
async def download_logs():
    """
    Download the complete central log file.
    """
    if not os.path.exists(CENTRAL_LOG_FILE):
        raise HTTPException(status_code=404, detail="Log file not found")
    
    return FileResponse(
        path=CENTRAL_LOG_FILE,
        filename=CENTRAL_LOG_FILE,
        media_type="text/plain"
    )

@app.get("/logs/tail")
async def tail_logs(lines: int = 50):
    """
    Get last N lines from central log as plain text.
    Perfect for real-time monitoring.
    """
    if not os.path.exists(CENTRAL_LOG_FILE):
        return PlainTextResponse("No logs available yet\n")
    
    try:
        with open(CENTRAL_LOG_FILE, "r") as f:
            all_lines = f.readlines()
            recent_lines = all_lines[-lines:] if len(all_lines) > lines else all_lines
        
        return PlainTextResponse("".join(recent_lines))
    except Exception as e:
        return PlainTextResponse(f"Error reading logs: {str(e)}\n")

# ---------------------------------------------------------
# 8. XAI INFERENCE ENDPOINT
# ---------------------------------------------------------

@app.post("/upload-xai")
async def upload_xai(file: UploadFile = File(...), client_name: str = None):
    """
    Predict from uploaded blood cell image + Grad-CAM + LIME explanation.
    
    Args:
        file: Uploaded image file
        client_name: Name of the client (model will be loaded from /content/drive/MyDrive/College/models/{client_name}.pt)
    
    Returns:
        JSON with prediction, confidence, and xAI visualizations (base64 encoded)
    """
    try:
        write_to_central_log(f"xAI Inference request received. Client: {client_name}, File: {file.filename}")
        
        # Validate client_name
        if not client_name:
            raise HTTPException(status_code=400, detail="client_name parameter is required")
        
        # Construct model path
        base_path = "/content/drive/MyDrive/College/models"
        model_path = f"{base_path}/{client_name}.pt"
        
        # Check if model exists
        if not os.path.exists(model_path):
            write_to_central_log(f"Model not found: {model_path}", level="ERROR")
            raise HTTPException(
                status_code=404, 
                detail=f"Model not found for client '{client_name}'. Expected path: {model_path}"
            )
        
        write_to_central_log(f"Loading model from: {model_path}")
        
        # Device setup
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        write_to_central_log(f"Using device: {device}")
        
        # ----------------------------------------------------
        # Load Model
        # ----------------------------------------------------
        checkpoint = torch.load(model_path, map_location=device)
        model_type = checkpoint.get('model_type', 'resnet18')
        num_classes = checkpoint.get('num_classes', 2)
        
        write_to_central_log(f"Model architecture: {model_type}, Classes: {num_classes}")
        
        # Build model architecture
        model = get_model(model_type, weights=None)
        model = modify_last_layer(model, num_classes)
        model.load_state_dict(checkpoint['state_dict'])
        model = model.to(device)
        model.eval()
        
        write_to_central_log(f"Model loaded successfully")
        
        # ----------------------------------------------------
        # Read and Preprocess Image
        # ----------------------------------------------------
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        write_to_central_log(f"Image loaded. Size: {image.size}")
        
        input_tensor = COMMON_TRANSFORM(image).unsqueeze(0).to(device)
        
        # ----------------------------------------------------
        # Inference
        # ----------------------------------------------------
        with torch.no_grad():
            outputs = model(input_tensor)
            probs = torch.softmax(outputs, dim=1)[0]
            pred_idx = int(torch.argmax(probs).item())
            confidence = float(probs[pred_idx].item())
        
        # CRITICAL: Class names MUST match ImageFolder alphabetical order
        # Dataset folders: 'all' (class 0), 'hem' (class 1)
        # Therefore: class 0 = ALL (Leukemia), class 1 = Healthy
        class_names = ["ALL (Leukemia)", "Healthy"]
        prediction = class_names[pred_idx]
        
        write_to_central_log(f"Prediction: {prediction} (confidence: {confidence:.4f})")
        
        # ----------------------------------------------------
        # Generate Grad-CAM
        # ----------------------------------------------------
        write_to_central_log("Generating Grad-CAM visualization...")
        target_layers = get_gradcam_target_layers(model, model_type)
        
        gradcam_base64 = generate_gradcam_base64(
            model=model,
            input_tensor=input_tensor,
            target_layers=target_layers,
            class_idx=pred_idx,
        )
        write_to_central_log("Grad-CAM generated successfully")
        
        # ----------------------------------------------------
        # Generate LIME
        # ----------------------------------------------------
        write_to_central_log("Generating LIME visualization...")
        lime_base64 = generate_lime_base64(
            model=model,
            input_tensor=input_tensor,
            class_idx=pred_idx,
            device=device,
            num_samples=1000,
        )
        write_to_central_log("LIME generated successfully")
        
        # ----------------------------------------------------
        # Response
        # ----------------------------------------------------
        response_data = {
            "prediction": prediction,
            "confidence": confidence,
            "class": pred_idx,
            "model": model_type,
            "model_path": model_path,
            "device": str(device),
            "client_name": client_name,
            "all_probabilities": {
                class_names[i]: float(probs[i].item()) 
                for i in range(len(class_names))
            },
            "xai": {
                "gradcam": {
                    "image_base64": gradcam_base64
                },
                "lime": {
                    "image_base64": lime_base64
                }
            },
        }
        
        write_to_central_log(f"xAI Inference completed successfully for {client_name}")
        
        return JSONResponse(status_code=200, content=response_data)

    except HTTPException:
        raise
    except Exception as e:
        error_msg = f"xAI Inference failed: {str(e)}"
        stack_trace = traceback.format_exc()
        write_to_central_log(error_msg, level="ERROR")
        write_to_central_log(f"Stack trace:\n{stack_trace}", level="ERROR")
        raise HTTPException(status_code=500, detail=str(e))

# ---------------------------------------------------------
# 9. Run Server with Error Handling
# ---------------------------------------------------------

AUTH_TOKEN = "37WqCiig3pb5s7D2WIizqmRnqnW_3nhckwCcX3J6gEaXB7paA"
ngrok.set_auth_token(AUTH_TOKEN)

try:
    http_tunnel = ngrok.connect(8000)
    public_url = http_tunnel.public_url

    print("\n" + "="*70)
    print("🚀 LOCAL TRAINING ORCHESTRATOR WITH XAI")
    print("="*70)
    print(f"🌐 Public URL: {public_url}")
    print(f"📝 Central Log: {CENTRAL_LOG_FILE}")
    print("\nAvailable Endpoints:")
    print(f"  📋 Status:        GET  {public_url}/")
    print(f"  👤 Enroll:        POST {public_url}/enroll_client")
    print(f"  🏋️  Train:         POST {public_url}/start_local_train")
    print(f"  📊 Client Status:  GET  {public_url}/train_status/{{client_name}}")
    print(f"  📜 Central Logs:   GET  {public_url}/logs?lines=100")
    print(f"  ⬇️  Download Logs:  GET  {public_url}/logs/download")
    print(f"  👁️  Tail Logs:      GET  {public_url}/logs/tail?lines=50")
    print(f"  🔬 XAI Inference:  POST {public_url}/upload-xai")
    print("="*70)
    print("\n📸 XAI Inference Usage:")
    print(f"  curl -X POST {public_url}/upload-xai \\")
    print(f"       -F 'file=@blood_cell.jpg' \\")
    print(f"       -F 'client_name=Hospital_A'")
    print("="*70)
    print(f"\n💡 Monitor logs: tail -f {CENTRAL_LOG_FILE}")
    print("="*70 + "\n")

    def run_uvicorn():
        uvicorn.run(app, port=8000, host="0.0.0.0")

    thread = threading.Thread(target=run_uvicorn)
    thread.start()

except PyngrokNgrokError as e:
    print("\n" + "="*70)
    print("❌ NGROK ERROR - Cannot Start Tunnel")
    print("="*70)
    print(f"\nError: {str(e)}\n")
    
    if "ERR_NGROK_108" in str(e) or "limited to 1 simultaneous" in str(e):
        print("🔴 PROBLEM: Free ngrok accounts only allow 1 tunnel at a time.\n")
        print("📋 SOLUTIONS:")
        print("  1. Stop the FL orchestrator (start_fl.ipynb) if it's running")
        print("  2. Run the cleanup cell above again")
        print("  3. Visit https://dashboard.ngrok.com/agents to see active sessions")
        print("  4. Upgrade to ngrok paid plan for multiple tunnels")
    else:
        print("💡 TIP: Run the cleanup cell above and try again")
    
    print("="*70 + "\n")
    raise

In [ ]:
import nest_asyncio
import uvicorn
import threading
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import get_model, list_models
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import os
import datetime
from pyngrok import ngrok
from pyngrok.process import kill_process
from fastapi import FastAPI, HTTPException, BackgroundTasks
from fastapi.responses import FileResponse, PlainTextResponse
from pydantic import BaseModel

# ---------------------------------------------------------
# 0. Setup & Cleanup
# ---------------------------------------------------------
try:
    ngrok.disconnect()
    kill_process()
except:
    pass

app = FastAPI()
nest_asyncio.apply()

# ---------------------------------------------------------
# 1. UNIFIED LOGGING SYSTEM
# ---------------------------------------------------------
CENTRAL_LOG_FILE = "local_training.log"
training_logs = {}

def write_to_central_log(message, level="INFO"):
    """
    Writes to the central local_training.log file.
    Similar to simulation_run.log for federated learning.
    """
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    formatted_msg = f"[{timestamp}] [{level}] {message}"
    
    # Write to central log file
    with open(CENTRAL_LOG_FILE, "a") as f:
        f.write(formatted_msg + "\n")
    
    # Also print to console
    print(formatted_msg)

def log_message(client_name, message, level="INFO"):
    """
    Enhanced logging that writes to:
    1. Central log file (local_training.log)
    2. Per-client log file ({client_name}.log)
    3. In-memory storage for API queries
    4. Console output
    """
    timestamp = datetime.datetime.now().strftime("%H:%M:%S")
    short_msg = f"[{timestamp}] {message}"
    full_msg = f"[{client_name}] {short_msg}"

    # 1. Write to CENTRAL log (like simulation_run.log)
    write_to_central_log(f"[{client_name}] {message}", level)

    # 2. Write to per-client log
    client_log_file = f"{client_name}.log"
    with open(client_log_file, "a") as f:
        f.write(short_msg + "\n")

    # 3. Update in-memory logs
    if client_name not in training_logs:
        training_logs[client_name] = {"status": "starting", "logs": []}
    training_logs[client_name]["logs"].append(short_msg)

# Initialize central log file
with open(CENTRAL_LOG_FILE, "w") as f:
    f.write(f"===== LOCAL TRAINING ORCHESTRATOR LOG =====\n")
    f.write(f"Started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"==========================================\n\n")

write_to_central_log("Local Training Orchestrator initialized")

# ---------------------------------------------------------
# 2. Data Models
# ---------------------------------------------------------

class ClientEnrollment(BaseModel):
    client_name: str
    model_type: str
    num_classes: int = 2

class TrainRequest(BaseModel):
    client_name: str
    dataset_path: str
    model_path: str
    epochs: int = 5
    batch_size: int = 16
    learning_rate: float = 0.001

# ---------------------------------------------------------
# 3. Model Architecture Helper
# ---------------------------------------------------------

def modify_last_layer(model, num_classes):
    if num_classes == 1000: return model

    if hasattr(model, 'fc') and isinstance(model.fc, nn.Linear):
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Linear):
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Sequential):
        updated = False
        for i in range(len(model.classifier) - 1, -1, -1):
            if isinstance(model.classifier[i], nn.Linear):
                model.classifier[i] = nn.Linear(model.classifier[i].in_features, num_classes)
                updated = True
                break
        if not updated:
             model.classifier.add_module(\"final_fc\", nn.Linear(1280, num_classes))
    return model

# ---------------------------------------------------------
# 4. Training Logic
# ---------------------------------------------------------

def run_training_background(client_name: str, dataset_path: str, model_path: str, epochs: int, batch_size: int, lr: float):
    """
    Background Task: Trains the model and logs to both central and per-client logs.
    """
    write_to_central_log(f"=== Training Started: {client_name} ===")
    log_message(client_name, f"🔄 Starting training for {model_path}...")
    log_message(client_name, f"Config: epochs={epochs}, batch_size={batch_size}, lr={lr}")
    training_logs[client_name]["status"] = "running"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log_message(client_name, f"⚙️ Device: {device}")

    try:
        # 1. Load Checkpoint
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model path {model_path} does not exist.")

        checkpoint = torch.load(model_path, map_location=device)
        model_name = checkpoint.get('model_type', 'resnet18')
        num_classes = checkpoint.get('num_classes', 2)

        # 2. Rebuild Model
        log_message(client_name, f"🏗️ Loading architecture: {model_name}")
        model = get_model(model_name, weights=None)
        model = modify_last_layer(model, num_classes)
        model.load_state_dict(checkpoint['state_dict'])
        model = model.to(device)

        # 3. Prepare Data
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        dataset = datasets.ImageFolder(root=dataset_path, transform=transform)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        log_message(client_name, f"📁 Dataset loaded: {len(dataset)} images, {len(dataloader)} batches")

        # 4. Training Loop
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        model.train()
        for epoch in range(epochs):
            running_loss = 0.0
            for i, (images, labels) in enumerate(dataloader):
                images, labels = images.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()

            # Log Epoch Stats
            avg_loss = running_loss / len(dataloader)
            log_message(client_name, f"📈 Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

        # 5. Save Updated Model
        checkpoint['state_dict'] = model.state_dict()
        torch.save(checkpoint, model_path)

        training_logs[client_name]["status"] = "completed"
        log_message(client_name, f"✅ Training Complete. Saved to {model_path}")
        write_to_central_log(f"=== Training Completed: {client_name} ===")

    except Exception as e:
        training_logs[client_name]["status"] = "failed"
        error_msg = f"❌ Training Failed: {str(e)}"
        log_message(client_name, error_msg, level="ERROR")
        write_to_central_log(f"=== Training Failed: {client_name} - {str(e)} ===", level="ERROR")

# ---------------------------------------------------------
# 5. API Endpoints
# ---------------------------------------------------------

@app.get("/")
def read_root():
    return {
        "status": "Server running",
        "service": "Local Training Orchestrator",
        "log_file": CENTRAL_LOG_FILE
    }

@app.post("/enroll_client")
async def enroll_client(client: ClientEnrollment):
    write_to_central_log(f"Client enrollment requested: {client.client_name} ({client.model_type})")
    try:
        model_name = client.model_type.lower().strip()
        available_models = list_models()
        if model_name not in available_models:
            raise HTTPException(status_code=400, detail="Model not found.")

        model = get_model(model_name, weights='DEFAULT')
        model = modify_last_layer(model, client.num_classes)

        base_path = "/content/drive/MyDrive/College/models"
        if not os.path.exists(base_path): 
            os.makedirs(base_path, exist_ok=True)
        full_path = f"{base_path}/{client.client_name}.pt"

        checkpoint = {
            'model_type': model_name,
            'num_classes': client.num_classes,
            'state_dict': model.state_dict()
        }
        torch.save(checkpoint, full_path)
        
        write_to_central_log(f"Client enrolled successfully: {client.client_name} -> {full_path}")
        return {"status": "success", "file_path": full_path}
    except Exception as e:
        write_to_central_log(f"Enrollment failed for {client.client_name}: {str(e)}", level="ERROR")
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/start_local_train")
async def start_local_train(request: TrainRequest, background_tasks: BackgroundTasks):
    """
    Starts training and initializes logging for the client.
    """
    write_to_central_log(f"Training request received for: {request.client_name}")
    write_to_central_log(f"  Dataset: {request.dataset_path}")
    write_to_central_log(f"  Model: {request.model_path}")
    write_to_central_log(f"  Config: epochs={request.epochs}, batch_size={request.batch_size}, lr={request.learning_rate}")

    if not os.path.exists(request.dataset_path):
        write_to_central_log(f"Dataset path not found: {request.dataset_path}", level="ERROR")
        raise HTTPException(status_code=404, detail="Dataset path not found.")

    # Initialize per-client log file
    log_file = f"{request.client_name}.log"
    with open(log_file, "w") as f:
        f.write(f"--- Training Log for {request.client_name} ---\n")
        f.write(f"Started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

    training_logs[request.client_name] = {"status": "pending", "logs": ["Request received."]}

    # Add to background
    background_tasks.add_task(
        run_training_background,
        request.client_name,
        request.dataset_path,
        request.model_path,
        request.epochs,
        request.batch_size,
        request.learning_rate
    )

    return {
        "status": "started",
        "message": f"Training started for {request.client_name}",
        "log_files": {
            "central": CENTRAL_LOG_FILE,
            "client": f"{request.client_name}.log"
        }
    }

@app.get("/train_status/{client_name}")
async def get_train_status(client_name: str):
    """
    Returns the logs and status for a specific client.
    """
    if client_name not in training_logs:
        raise HTTPException(status_code=404, detail="No training logs found for this client.")

    return training_logs[client_name]

@app.get("/logs")
async def get_central_logs(lines: int = 100):
    """
    Fetch recent lines from the central local_training.log file.
    Similar to tailing simulation_run.log.
    
    Args:
        lines: Number of recent lines to return (default: 100)
    """
    if not os.path.exists(CENTRAL_LOG_FILE):
        return {"logs": [], "message": "No logs available yet"}
    
    try:
        with open(CENTRAL_LOG_FILE, "r") as f:
            all_lines = f.readlines()
            recent_lines = all_lines[-lines:] if len(all_lines) > lines else all_lines
        
        return {
            "logs": [line.strip() for line in recent_lines],
            "total_lines": len(all_lines),
            "returned_lines": len(recent_lines)
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to read logs: {str(e)}")

@app.get("/logs/download")
async def download_logs():
    """
    Download the complete central log file.
    """
    if not os.path.exists(CENTRAL_LOG_FILE):
        raise HTTPException(status_code=404, detail="Log file not found")
    
    return FileResponse(
        path=CENTRAL_LOG_FILE,
        filename=CENTRAL_LOG_FILE,
        media_type="text/plain"
    )

@app.get("/logs/tail")
async def tail_logs(lines: int = 50):
    """
    Get last N lines from central log as plain text.
    Perfect for real-time monitoring.
    """
    if not os.path.exists(CENTRAL_LOG_FILE):
        return PlainTextResponse("No logs available yet\n")
    
    try:
        with open(CENTRAL_LOG_FILE, "r") as f:
            all_lines = f.readlines()
            recent_lines = all_lines[-lines:] if len(all_lines) > lines else all_lines
        
        return PlainTextResponse("".join(recent_lines))
    except Exception as e:
        return PlainTextResponse(f"Error reading logs: {str(e)}\n")

# ---------------------------------------------------------
# 6. Run Server
# ---------------------------------------------------------

AUTH_TOKEN = "37WqCiig3pb5s7D2WIizqmRnqnW_3nhckwCcX3J6gEaXB7paA"
ngrok.set_auth_token(AUTH_TOKEN)
http_tunnel = ngrok.connect(8000)
public_url = http_tunnel.public_url

print("\n" + "="*70)
print("🚀 LOCAL TRAINING ORCHESTRATOR")
print("="*70)
print(f"🌐 Public URL: {public_url}")
print(f"📝 Central Log: {CENTRAL_LOG_FILE}")
print("\nAvailable Endpoints:")
print(f"  📋 Status:       GET  {public_url}/")
print(f"  👤 Enroll:       POST {public_url}/enroll_client")
print(f"  🏋️  Train:        POST {public_url}/start_local_train")
print(f"  📊 Client Status: GET  {public_url}/train_status/{{client_name}}")
print(f"  📜 Central Logs:  GET  {public_url}/logs?lines=100")
print(f"  ⬇️  Download Logs: GET  {public_url}/logs/download")
print(f"  👁️  Tail Logs:     GET  {public_url}/logs/tail?lines=50")
print("="*70)
print(f"\n💡 Monitor logs: tail -f {CENTRAL_LOG_FILE}")
print("="*70 + "\n")

def run_uvicorn():
    uvicorn.run(app, port=8000, host="0.0.0.0")

thread = threading.Thread(target=run_uvicorn)
thread.start()

# 🔬 **XAI Inference Endpoint Usage**

## New Endpoint: `/upload-xai`

This endpoint performs inference on blood cell images and generates explainable AI visualizations (Grad-CAM and LIME).

### **Features**
- ✅ Load any client model from `/content/drive/MyDrive/College/models`
- ✅ Predict ALL (Acute Lymphoblastic Leukemia) diagnosis
- ✅ Generate Grad-CAM heatmap (attention visualization)
- ✅ Generate LIME explanation (super-pixel importance)
- ✅ Return base64-encoded images for easy display

### **Python Usage Example**

```python
import requests
import base64
from PIL import Image
from io import BytesIO

# Your ngrok URL from the server output above
ngrok_url = "https://YOUR-NGROK-URL.ngrok-free.app"

# Upload image for inference
with open("blood_cell_sample.jpg", "rb") as f:
    files = {"file": f}
    data = {"client_name": "Hospital_A"}  # Use your client name
    
    response = requests.post(
        f"{ngrok_url}/upload-xai",
        files=files,
        data=data
    )

result = response.json()

# Display prediction
print(f"Prediction: {result['prediction']}")
print(f"Confidence: {result['confidence']:.4f}")
print(f"Model: {result['model']}")

# Decode and display Grad-CAM
gradcam_img = Image.open(BytesIO(base64.b64decode(result['xai']['gradcam']['image_base64'])))
gradcam_img.show()

# Decode and display LIME
lime_img = Image.open(BytesIO(base64.b64decode(result['xai']['lime']['image_base64'])))
lime_img.show()
```

### **curl Usage Example**

```bash
# Replace with your ngrok URL and client name
curl -X POST https://YOUR-NGROK-URL.ngrok-free.app/upload-xai \
  -F "file=@/path/to/blood_cell.jpg" \
  -F "client_name=Hospital_A" \
  | jq .
```

### **Response Format**

```json
{
  "prediction": "ALL (Leukemia)",
  "confidence": 0.87,
  "class": 1,
  "model": "resnet18",
  "model_path": "/content/drive/MyDrive/College/models/Hospital_A.pt",
  "device": "cuda",
  "client_name": "Hospital_A",
  "all_probabilities": {
    "Healthy": 0.13,
    "ALL (Leukemia)": 0.87
  },
  "xai": {
    "gradcam": {
      "image_base64": "iVBORw0KGgoAAAANS..."
    },
    "lime": {
      "image_base64": "iVBORw0KGgoAAAANS..."
    }
  }
}
```

### **Important Notes**

1. **Client Name**: The `client_name` parameter must match a model file in `/content/drive/MyDrive/College/models/{client_name}.pt`
2. **Performance**: LIME generation takes 10-15 seconds (runs 1000 forward passes for explanation)
3. **Supported Models**: ResNet, MobileNet, EfficientNet, DenseNet, VGG (automatically detects target layers)
4. **Image Format**: Accepts JPG, PNG, and other common formats
5. **Logging**: All inference requests are logged to `local_training.log`

### **Integration with Backend**

You can update your backend's `federated.py` to proxy to this endpoint:

```python
# In backend/app/routes/federated.py
INFERENCE_URL = os.getenv("INFERENCE_URL", "http://localhost:8001")

@router.post("/inference-xai")
async def inference_xai(file: UploadFile, client_name: str):
    files = {"file": file.file}
    data = {"client_name": client_name}
    response = requests.post(f"{INFERENCE_URL}/upload-xai", files=files, data=data)
    return response.json()
```

# **Monitor Central Log (Like simulation_run.log)**

In [ ]:
# Tail the central log file to see ALL training activities
!tail -f local_training.log

# **Monitor Specific Client Log**

In [ ]:
# Monitor a specific client's training log
# Replace 'CLIENT_NAME' with actual client name
!tail -f CLIENT_NAME.log

# **View Recent Logs (Without Blocking)**

In [ ]:
# View last 50 lines of central log (non-blocking)
!tail -n 50 local_training.log

# **🔍 Debug: Check if Training is Actually Running**

In [ ]:
# Run this if training appears stuck
import subprocess
import psutil
import os

print("🔍 DIAGNOSTICS\n")

# 1. Check if Python process is using CPU/GPU
print("1. Python Processes:")
for proc in psutil.process_iter(['pid', 'name', 'cpu_percent', 'memory_percent']):
    if 'python' in proc.info['name'].lower():
        print(f"   PID {proc.info['pid']}: CPU={proc.info['cpu_percent']:.1f}%, MEM={proc.info['memory_percent']:.1f}%")

# 2. Check GPU usage (if available)
print("\n2. GPU Status:")
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except:
    print("   No GPU or nvidia-smi not available")

# 3. Check if training thread is alive
print("\n3. Active Threads:")
import threading
for thread in threading.enumerate():
    print(f"   {thread.name}: alive={thread.is_alive()}")

# 4. View last few log lines
print("\n4. Recent Log Lines:")
try:
    with open('local_training.log', 'r') as f:
        lines = f.readlines()
        for line in lines[-10:]:
            print(f"   {line.strip()}")
except:
    print("   No log file found")

print("\n💡 If CPU is at 0% and no errors, the training is HUNG. You need to restart.")

# **Cleanup: Kill Port 8000**

In [ ]:
# Kill any process running on port 8000
!fuser -k 8000/tcp
print("Attempted to kill processes on port 8000.")